In [2]:
import sys
project_root = "/home/krosen5/Court Case Sequence Transformer Model/court_case_transformer_project"

if project_root not in sys.path:
    sys.path.insert(0, project_root)

print(sys.path[:3])

['/home/krosen5/Court Case Sequence Transformer Model/court_case_transformer_project', '/home/krosen5/anaconda3/envs/legalsequencetransformer/lib/python313.zip', '/home/krosen5/anaconda3/envs/legalsequencetransformer/lib/python3.13']


In [3]:
import os
print(os.getcwd())
print(os.listdir("."))

/home/krosen5/Court Case Sequence Transformer Model/court_case_transformer_project/side_tests
['load_in_model_just_clustering.ipynb', 'load_in_model_just_confident_misclassifications.ipynb', '__init__.py', '.ipynb_checkpoints', '__pycache__', 'load_in_model_complete.ipynb']


In [5]:
import sys
import pickle
import torch
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from contextlib import contextmanager
from pathlib import Path
from sklearn.model_selection import KFold

from utils.cache_directory_utils import CacheDirectoryManager
from data.load_data import mount_drive, load_data
from models.transformer_model import CaseTransformerModel
from models.transformer_creation import create_vocabularies_from_data
from config import get_model_config
from notebook_helpers import prepare_sequences_and_labels
from notebook_helpers import (
    run_confident_misclassification_analysis,
    analyze_confident_misclass_for_label,
)

In [6]:
parameters_dict = {
    'model_validation_strategy': 'k_fold_cross_validation',
    'k_fold_num_folds': 5,
    'k_fold_shuffle': True,
    'random_seed': 42,
    'skip_duplicate_labels': False,
    'skip_duplicate_labels_on_import': False,
    'bert_model_to_use': 'bert-base-uncased',
    'bert_batch_size': 32,
    'sampling_strategy': 'balanced',
    'weighted_loss_function': False,
    'use_mixed_precision': True,
    'use_learning_rate_scheduler': True,
    'gradient_clip_norm': 1.0,
    'number_of_epochs_for_training': 20,
    'batch_size_for_training': 16,
    'low_frequency_threshold': 400,
    'folder_name_directory': 'balanced_8',
    'minimum_confidence_for_wrong_prediction_inclusion': 0.8,
    'number_of_highest_probability_predictions_per_step': 5,
    'maximum_number_of_confident_mistakes_to_return': 25,
    'maximum_validation_sequences_to_process_for_analysis_of_incorrect': 100,
}

In [7]:
@contextmanager
def log_to_file(log_path):
    """Redirect stdout to a log file only (nothing printed to console)."""
    log_path = Path(log_path)
    log_path.parent.mkdir(parents=True, exist_ok=True)

    with open(log_path, 'w') as f:
        old_stdout = sys.stdout
        sys.stdout = f
        try:
            yield
        finally:
            sys.stdout = old_stdout


@contextmanager
def capture_plots(save_dir, prefix='plot'):
    """
    Switches to Agg backend so plt.show() is a no-op,
    then after the block saves every new figure to save_dir.
    Restores the original backend afterwards.
    """
    save_dir = Path(save_dir)
    save_dir.mkdir(parents=True, exist_ok=True)

    original_backend = matplotlib.get_backend()
    matplotlib.use('Agg')

    # snapshot which figures already exist before the call
    existing_figs = set(plt.get_fignums())

    try:
        yield
    finally:
        # save any figures created inside the block
        new_figs = [n for n in plt.get_fignums() if n not in existing_figs]
        for i, fig_num in enumerate(new_figs):
            fig = plt.figure(fig_num)
            out_path = save_dir / f'{prefix}_{i}.png'
            fig.savefig(out_path, dpi=150, bbox_inches='tight')
            plt.close(fig)

        # restore original backend
        matplotlib.use(original_backend)


In [8]:
# ═══════════════════════════════════════════════════════════
# 1. Config
# ═══════════════════════════════════════════════════════════
 
BASE_CACHE       = Path('/data/krosen5/cache/transformers/balanced_8')
MODELS_DIR       = BASE_CACHE / 'saved_models'
MODEL_RESULTS_PATH = MODELS_DIR / 'model_results.pkl'
OUTPUT_DIR       = BASE_CACHE / 'confident_misclass_by_label'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
 
N_FOLDS = 5
 
# ═══════════════════════════════════════════════════════════
# 2. Cache manager + label processor
# ═══════════════════════════════════════════════════════════
 
model_cache_manager = CacheDirectoryManager(base_path=BASE_CACHE)
 
label_processor_path = MODELS_DIR / 'label_processor.pkl'
with open(label_processor_path, 'rb') as f:
    label_processor = pickle.load(f)
print(f'✅ Loaded label processor ({len(label_processor.label_names)} labels)')
 
# ═══════════════════════════════════════════════════════════
# 3. Load model_results and extract confident misclass results
# ═══════════════════════════════════════════════════════════
 
print(f'Loading model results from {MODEL_RESULTS_PATH}...')
with open(MODEL_RESULTS_PATH, 'rb') as f:
    model_results = pickle.load(f)
 
confident_misclass_results_per_fold = model_results.get('confident_misclass_results_per_fold')
 
if confident_misclass_results_per_fold is None:
    raise ValueError(
        'confident_misclass_results_per_fold not found in model_results.pkl. '
        'Make sure the analysis was run and saved before using this script.'
    )
 
n_folds_found = len(confident_misclass_results_per_fold)
print(f'✅ Loaded confident misclassification results for {n_folds_found} folds')
 
# ═══════════════════════════════════════════════════════════
# 4. Per-label breakdown — logs + plots saved per fold
# ═══════════════════════════════════════════════════════════
 
print(f'\n{"═"*60}')
print('  Per-label breakdown')
print(f'{"═"*60}\n')
 
n_labels = len(label_processor.label_names)
 
for label_num, label_name in enumerate(label_processor.label_names, start=1):
    label_dir = OUTPUT_DIR / label_name
 
    for fold_idx in range(n_folds_found):
        fold_dir = label_dir / f'fold_{fold_idx + 1}'
 
        with log_to_file(fold_dir / 'output.log'), \
             capture_plots(fold_dir, prefix='misclass_plot'):
            analyze_confident_misclass_for_label(
                confident_misclass_results_per_fold[fold_idx],
                target_label=label_name,
            )
 
    print(f'  ✅ [{label_num}/{n_labels}] {label_name} done → {label_dir}')
 
print(f'\n✅ Done. All results saved under {OUTPUT_DIR}')

✅ Loaded label processor (20 labels)
Loading model results from /data/krosen5/cache/transformers/balanced_8/saved_models/model_results.pkl...
✅ Loaded confident misclassification results for 5 folds

════════════════════════════════════════════════════════════
  Per-label breakdown
════════════════════════════════════════════════════════════

  ✅ [1/20] FIRST_APPEARANCE_HEARING_OCCURRED done → /data/krosen5/cache/transformers/balanced_8/confident_misclass_by_label/FIRST_APPEARANCE_HEARING_OCCURRED
  ✅ [2/20] BOND_SET done → /data/krosen5/cache/transformers/balanced_8/confident_misclass_by_label/BOND_SET
  ✅ [3/20] BOND_NOT_SET done → /data/krosen5/cache/transformers/balanced_8/confident_misclass_by_label/BOND_NOT_SET
  ✅ [4/20] DEFENDANT_RELEASED_ON_OWN_RECOGNIZANCE done → /data/krosen5/cache/transformers/balanced_8/confident_misclass_by_label/DEFENDANT_RELEASED_ON_OWN_RECOGNIZANCE
  ✅ [5/20] THE_COURT_APPOINTED_THE_ORLEANS_PUBLIC_DEFENDER done → /data/krosen5/cache/transformers/balanc